This notebook contains a tutorial for how to use the open source model-diffing crosscoders from https://huggingface.co/ckkissane/crosscoder-gemma-2-2b-model-diff

It shows how to load  the crosscoder weights, replicate [Anthropic's core results](https://transformer-circuits.pub/2024/crosscoders/index.html#model-diffing), implement evals, and generate latent dashboards with a [fork of sae_vis](https://github.com/ckkissane/sae_vis/tree/crosscoder-vis).

# Setup

In [ ]:
!pip install transformer_lens

In [1]:
import torch
from torch import nn
import pprint
import torch.nn.functional as F
from typing import Optional, Union
from huggingface_hub import hf_hub_download, notebook_login
import json
import einops
import plotly.express as px

from typing import NamedTuple

## loading the models

In [2]:
from transformer_lens import HookedTransformer

In [ ]:
notebook_login()

In [ ]:
# !pip install -q transformers huggingface_hub


In [ ]:
!pip install --upgrade transformer_lens


In [ ]:
pip install --upgrade transformers


The crosscoder was trained to model-diff Gemma-2 2b base and IT models, so we'll load these with TransformerLens. I use an A100 with colab pro. This might be too memory intensive for smaller GPUs.

In [ ]:
device = 'cuda:0'
torch.set_grad_enabled(False) # important for memory

base_model = HookedTransformer.from_pretrained(
    "gemma-2-2b",
    device=device,
    dtype=torch.bfloat16
)

chat_model = HookedTransformer.from_pretrained(
    "gemma-2-2b-it",
    device=device,
    dtype=torch.bfloat16
)

## loading the crosscoder

This is implementation of the crosscoder, basically copied from https://github.com/ckkissane/crosscoder-model-diff-replication

In [6]:
import os

In [7]:
DTYPES = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}

class LossOutput(NamedTuple):
    # loss: torch.Tensor
    l2_loss: torch.Tensor
    l1_loss: torch.Tensor
    l0_loss: torch.Tensor
    explained_variance: torch.Tensor
    explained_variance_A: torch.Tensor
    explained_variance_B: torch.Tensor

class CrossCoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d_hidden = self.cfg["dict_size"]
        d_in = self.cfg["d_in"]
        self.dtype = DTYPES[self.cfg["enc_dtype"]]
        torch.manual_seed(self.cfg["seed"])
        # hardcoding n_models to 2
        self.W_enc = nn.Parameter(
            torch.empty(2, d_in, d_hidden, dtype=self.dtype)
        )
        self.W_dec = nn.Parameter(
            torch.nn.init.normal_(
                torch.empty(
                    d_hidden, 2, d_in, dtype=self.dtype
                )
            )
        )

        # Make norm of W_dec 0.1 for each column, separate per layer
        self.W_dec.data = (
            self.W_dec.data / self.W_dec.data.norm(dim=-1, keepdim=True) * self.cfg["dec_init_norm"]
        )
        # Initialise W_enc to be the transpose of W_dec
        self.W_enc.data = einops.rearrange(
            self.W_dec.data.clone(),
            "d_hidden n_models d_model -> n_models d_model d_hidden",
        )
        self.b_enc = nn.Parameter(torch.zeros(d_hidden, dtype=self.dtype))
        self.b_dec = nn.Parameter(
            torch.zeros((2, d_in), dtype=self.dtype)
        )
        self.d_hidden = d_hidden

        self.to(self.cfg["device"])
        self.save_dir = None
        self.save_version = 0

    def encode(self, x, apply_relu=True):
        # x: [batch, n_models, d_model]
        x_enc = einops.einsum(
            x,
            self.W_enc,
            "batch n_models d_model, n_models d_model d_hidden -> batch d_hidden",
        )
        if apply_relu:
            acts = F.relu(x_enc + self.b_enc)
        else:
            acts = x_enc + self.b_enc
        return acts

    def decode(self, acts):
        # acts: [batch, d_hidden]
        acts_dec = einops.einsum(
            acts,
            self.W_dec,
            "batch d_hidden, d_hidden n_models d_model -> batch n_models d_model",
        )
        return acts_dec + self.b_dec

    def forward(self, x):
        # x: [batch, n_models, d_model]
        acts = self.encode(x)
        return self.decode(acts)

    def get_losses(self, x):
        # x: [batch, n_models, d_model]
        x = x.to(self.dtype)
        acts = self.encode(x)
        # acts: [batch, d_hidden]
        x_reconstruct = self.decode(acts)
        diff = x_reconstruct.float() - x.float()
        squared_diff = diff.pow(2)
        l2_per_batch = einops.reduce(squared_diff, 'batch n_models d_model -> batch', 'sum')
        l2_loss = l2_per_batch.mean()

        total_variance = einops.reduce((x - x.mean(0)).pow(2), 'batch n_models d_model -> batch', 'sum')
        explained_variance = 1 - l2_per_batch / total_variance

        per_token_l2_loss_A = (x_reconstruct[:, 0, :] - x[:, 0, :]).pow(2).sum(dim=-1).squeeze()
        total_variance_A = (x[:, 0, :] - x[:, 0, :].mean(0)).pow(2).sum(-1).squeeze()
        explained_variance_A = 1 - per_token_l2_loss_A / total_variance_A

        per_token_l2_loss_B = (x_reconstruct[:, 1, :] - x[:, 1, :]).pow(2).sum(dim=-1).squeeze()
        total_variance_B = (x[:, 1, :] - x[:, 1, :].mean(0)).pow(2).sum(-1).squeeze()
        explained_variance_B = 1 - per_token_l2_loss_B / total_variance_B

        decoder_norms = self.W_dec.norm(dim=-1)
        # decoder_norms: [d_hidden, n_models]
        total_decoder_norm = einops.reduce(decoder_norms, 'd_hidden n_models -> d_hidden', 'sum')
        l1_loss = (acts * total_decoder_norm[None, :]).sum(-1).mean(0)

        l0_loss = (acts>0).float().sum(-1).mean()

        return LossOutput(l2_loss=l2_loss, l1_loss=l1_loss, l0_loss=l0_loss, explained_variance=explained_variance, explained_variance_A=explained_variance_A, explained_variance_B=explained_variance_B)


    @classmethod
    def load_from_hf(
        cls,
        repo_id: str = "ckkissane/crosscoder-gemma-2-2b-model-diff",
        path: str = "blocks.14.hook_resid_pre",
        device: Optional[Union[str, torch.device]] = None
    ) -> "CrossCoder":
        """
        Load CrossCoder weights and config from HuggingFace.

        Args:
            repo_id: HuggingFace repository ID
            path: Path within the repo to the weights/config
            model: The transformer model instance needed for initialization
            device: Device to load the model to (defaults to cfg device if not specified)

        Returns:
            Initialized CrossCoder instance
        """

        # Download config and weights
        config_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{path}/cfg.json"
        )
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{path}/cc_weights.pt"
        )

        # Load config
        with open(config_path, 'r') as f:
            cfg = json.load(f)

        # Override device if specified
        if device is not None:
            cfg["device"] = str(device)

        # Initialize CrossCoder with config
        instance = cls(cfg)

        # Load weights
        state_dict = torch.load(weights_path, map_location=cfg["device"])
        instance.load_state_dict(state_dict)

        return instance


class MatryoshkaCrossCoderV2(CrossCoder):
    def __init__(self, cfg):
        # Overwrite the flat dictionary size with the sum over group_sizes.
        total_hidden = sum(cfg["group_sizes"])  # total_hidden becomes, e.g., 4096+4096+8192
        cfg = dict(cfg)
        cfg["dict_size"] = total_hidden         # Override flat dict size with total nested size
        super().__init__(cfg)  # This will now create W_enc of shape [2, d_in, total_hidden]

        # Save the group configuration and compute group boundaries (indices)
        self.group_sizes = cfg["group_sizes"]
        # e.g. group_indices = [0, 4096, 4096+4096, 4096+4096+8192]
        self.group_indices = [0] + list(torch.tensor(self.group_sizes).cumsum(dim=0).tolist())

    def decode_nested(self, acts):
        """
        Computes intermediate reconstructions over nested groups.
        Returns a list of reconstructions, where the final one is the sum of all groups.
        """
        batch = acts.shape[0]
        # Start with the bias (broadcasted to [batch, 2, d_model])
        current_reconstruction = self.b_dec.unsqueeze(0).expand(batch, -1, -1)
        nested_recs = []
        # Loop over each group defined by self.group_indices
        for i in range(len(self.group_sizes)):
            start = self.group_indices[i]
            end = self.group_indices[i+1]
            # Slice the latent activations for the current group.
            acts_group = acts[:, start:end]  # shape: [batch, group_size]
            # And select the corresponding decoder slice (shape: [group_size, 2, d_model])
            W_dec_group = self.W_dec[start:end]
            # Compute this group’s reconstruction contribution:
            rec_group = einops.einsum(acts_group, W_dec_group, "batch group_size, group_size n_models d_model -> batch n_models d_model")
            # Add it to the accumulated reconstruction:
            current_reconstruction = current_reconstruction + rec_group
            nested_recs.append(current_reconstruction)
        return nested_recs

    def decode(self, acts):
        # For compatibility, return the final reconstruction (using all groups)
        nested_recs = self.decode_nested(acts)
        return nested_recs[-1]

    def get_losses(self, x):
        x = x.to(self.dtype)
        acts = self.encode(x)  # shape: [batch, total_hidden]
        nested_recs = self.decode_nested(acts)  # list of [batch, n_models, d_model]

        # Compute l2 loss for each nested reconstruction and sum them.
        l2_losses = []
        for rec in nested_recs:
            diff = rec.float() - x.float()
            squared_diff = diff.pow(2)
            l2_loss_group = squared_diff.sum(dim=[1,2]).mean()
            l2_losses.append(l2_loss_group)
        l2_loss = sum(l2_losses)  # you may want to weight each group differently

        # (Optional) Reuse the same computation of l1_loss and l0_loss as before.
        decoder_norms = self.W_dec.norm(dim=-1)  # shape: [total_hidden, n_models]
        total_decoder_norm = decoder_norms.sum(dim=1)
        l1_loss = (acts * total_decoder_norm[None, :]).sum(-1).mean()
        l0_loss = (acts > 0).float().sum(-1).mean()

        # Compute explained variance on the final reconstruction
        final_rec = nested_recs[-1]
        diff_final = final_rec.float() - x.float()
        l2_final = diff_final.pow(2).sum(dim=[1,2])
        total_var = (x - x.mean(0)).pow(2).sum(dim=[1,2])
        explained_variance = 1 - l2_final / total_var

        # For per-model variance (A/B), repeat as needed…
        # (See the original implementation for per-model explained variance.)
        # Compute explained variance for model A
        per_token_l2_loss_A = (final_rec[:, 0, :] - x[:, 0, :]).pow(2).sum(dim=-1).squeeze()
        total_variance_A = (x[:, 0, :] - x[:, 0, :].mean(0)).pow(2).sum(dim=-1).squeeze()
        explained_variance_A = 1 - per_token_l2_loss_A / total_variance_A

        # Compute explained variance for model B
        per_token_l2_loss_B = (final_rec[:, 1, :] - x[:, 1, :]).pow(2).sum(dim=-1).squeeze()
        total_variance_B = (x[:, 1, :] - x[:, 1, :].mean(0)).pow(2).sum(dim=-1).squeeze()
        explained_variance_B = 1 - per_token_l2_loss_B / total_variance_B

        return LossOutput(
            l2_loss=l2_loss,
            l1_loss=l1_loss,
            l0_loss=l0_loss,
            explained_variance=explained_variance,
            explained_variance_A=explained_variance_A,  # placeholder
            explained_variance_B=explained_variance_B   # placeholder
        )

    import os

    @classmethod
    def load_from_hf(
        cls,
        repo_id: str = "oanaflores/crosscoder-gemma-2-2b-model-diff-matryoshka-v2",
        path: str = "blocks.14.hook_resid_pre",
        device: Optional[Union[str, torch.device]] = None
    ) -> "MatryoshkaCrossCoderV2":
        """
        Load MatryoshkaCrossCoderV2 weights and config from HuggingFace (fixed).
        """

        # Download config and weights
        config_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{path}/5_cfg.json"
        )
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{path}/5.pt"
        )

        # Load config
        with open(config_path, 'r') as f:
            cfg = json.load(f)

        # Override device if specified
        if device is not None:
            cfg["device"] = str(device)

        # Initialize CrossCoder with config
        instance = cls(cfg)

        # Load weights
        state_dict = torch.load(weights_path, map_location=cfg["device"])
        instance.load_state_dict(state_dict)

        return instance



Before analyzing the crosscoder, we need to load the trained crosscoder weights from huggingface https://huggingface.co/ckkissane/crosscoder-gemma-2-2b-model-diff

In [ ]:
cross_coder = CrossCoder.load_from_hf()
cross_coder

# Replicating Anthropic results

This section replicates the key results from Anthropic. We'll first analyze the relative norms between the base vs IT decoder vectors.

In [ ]:
norms = cross_coder.W_dec.norm(dim=-1)
norms.shape

In [ ]:
relative_norms = norms[:, 1] / norms.sum(dim=-1)
relative_norms.shape

In [ ]:
fig = px.histogram(
    relative_norms.detach().cpu().numpy(),
    title="Gemma 2 2B Base vs IT Model Diff",
    labels={"value": "Relative decoder norm strength"},
    nbins=200,
)

fig.update_layout(showlegend=False)
fig.update_yaxes(title_text="Number of Latents")

# Update x-axis ticks
fig.update_xaxes(
    tickvals=[0, 0.25, 0.5, 0.75, 1.0],
    ticktext=['0', '0.25', '0.5', '0.75', '1.0']
)

fig.show()

We notice 3 main clusters, replicating Anthropic's result:
* base specific latents (left)
* IT specific latents (right)
* shared latents (middle)

Now let's check the cosine similarity of the "shared" decoder vectors between both models:

In [ ]:
shared_latent_mask = (relative_norms < 0.7) & (relative_norms > 0.3)
shared_latent_mask.shape

In [ ]:
cosine_sims = (cross_coder.W_dec[:, 0, :] * cross_coder.W_dec[:, 1, :]).sum(dim=-1) / (cross_coder.W_dec[:, 0, :].norm(dim=-1) * cross_coder.W_dec[:, 1, :].norm(dim=-1))
cosine_sims.shape

In [ ]:
fig = px.histogram(
    cosine_sims[shared_latent_mask].to(torch.float32).detach().cpu().numpy(),
    #title="Cosine similarity of decoder vectors between models",
    log_y=True,  # Sets the y-axis to log scale
    range_x=[-1, 1],  # Sets the x-axis range from -1 to 1
    nbins=100,  # Adjust this value to change the number of bins
    labels={"value": "Cosine similarity of decoder vectors between models"}
)

fig.update_layout(showlegend=False)
fig.update_yaxes(title_text="Number of Latents (log scale)")

fig.show()

We notice very high alignment, with a few outliers with low (or even negative) cosine sim. This corroborates the result from Anthropic's paper.

# CE Loss Evals

This section provides some code to start evaluating the reconstruction fidelity of the crosscoder. We can check how replacing both model's activations with their cross-coded reconstructions affects cross entropy loss. This is a common practice in SAE evals, but is a bit more involved with crosscoders.


We first need to load in the dataset. We trained the crosscoder on 50% pile text, and 50% LmSys. We pretokenized this dataset and stored it on HF at https://huggingface.co/datasets/ckkissane/pile-lmsys-mix-1m-tokenized-gemma-2 .


In [ ]:
from datasets import load_dataset
def load_pile_lmsys_mixed_tokens():
    try:
        print("Loading data from disk")
        all_tokens = torch.load("/workspace/data/pile-lmsys-mix-1m-tokenized-gemma-2.pt")
    except:
        print("Data is not cached. Loading data from HF")
        data = load_dataset(
            "ckkissane/pile-lmsys-mix-1m-tokenized-gemma-2",
            split="train",
            cache_dir="/workspace/cache/"
        )
        data.save_to_disk("/workspace/data/pile-lmsys-mix-1m-tokenized-gemma-2.hf")
        data.set_format(type="torch", columns=["input_ids"])
        all_tokens = data["input_ids"]
        torch.save(all_tokens, "/workspace/data/pile-lmsys-mix-1m-tokenized-gemma-2.pt")
        print(f"Saved tokens to disk")
    return all_tokens

all_tokens = load_pile_lmsys_mixed_tokens()

When we trained our crosscoder, we normalized both the base and chat model activations such that they both have avg norm sqrt(d_model). In training, this is implemented by estimating scaling constants such that norm(scale * act) = sqrt(d_model) over a subset of the training distribution. I'll just hard code them in this demo.


This means we also need to normalize the activations during analysis. Further, since we'll be splicing the reconstructed activations back into the forward pass of the model, we need to "unscale" the reconstructed activations too. We can alternatively fold this into the weights, as below:


In [18]:
import copy
folded_cross_coder = copy.deepcopy(cross_coder)


def fold_activation_scaling_factor(cross_coder, base_scaling_factor, chat_scaling_factor):
    cross_coder.W_enc.data[0, :, :] = cross_coder.W_enc.data[0, :, :] * base_scaling_factor
    cross_coder.W_enc.data[1, :, :] = cross_coder.W_enc.data[1, :, :] * chat_scaling_factor

    cross_coder.W_dec.data[:, 0, :] = cross_coder.W_dec.data[:, 0, :] / base_scaling_factor
    cross_coder.W_dec.data[:, 1, :] = cross_coder.W_dec.data[:, 1, :] / chat_scaling_factor

    cross_coder.b_dec.data[0, :] = cross_coder.b_dec.data[0, :] / base_scaling_factor
    cross_coder.b_dec.data[1, :] = cross_coder.b_dec.data[1, :] / chat_scaling_factor
    return cross_coder

base_estimated_scaling_factor = 0.2758961493232058
chat_estimated_scaling_factor = 0.24422852496546169
folded_cross_coder = fold_activation_scaling_factor(folded_cross_coder, base_estimated_scaling_factor, chat_estimated_scaling_factor)
folded_cross_coder = folded_cross_coder.to(torch.bfloat16)

This code implements the "splicing" of crosscoder reconstructions into both model's forward pass, and measures its effect on cross entropy loss. It's a bit more involved than SAEs, since crosscoders require the concatentation of both model's activations as input. We'll only do one small batch since colab memory is scarce, but in practice it's better to average over multiple examples.

In [19]:
from functools import partial

def splice_act_hook(act, hook, spliced_act):
    act[:, 1:, :] = spliced_act # Drop BOS
    return act

def zero_ablation_hook(act, hook):
    act[:] = 0
    return act

def get_ce_recovered_metrics(tokens, model_A, model_B, cross_coder):
    # get clean loss
    ce_clean_A = model_A(tokens, return_type="loss")
    ce_clean_B = model_B(tokens, return_type="loss")

    # get zero abl loss
    ce_zero_abl_A = model_A.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks = [(cross_coder.cfg["hook_point"], zero_ablation_hook)],
    )
    ce_zero_abl_B = model_B.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks = [(cross_coder.cfg["hook_point"], zero_ablation_hook)],
    )

    # bunch of annoying set up for splicing
    _, cache_A = model_A.run_with_cache(
        tokens,
        names_filter=cross_coder.cfg["hook_point"],
        return_type=None,
        )
    resid_act_A = cache_A[cross_coder.cfg["hook_point"]]

    _, cache_B = model_B.run_with_cache(
        tokens,
        names_filter=cross_coder.cfg["hook_point"],
        return_type=None,
        )
    resid_act_B = cache_B[cross_coder.cfg["hook_point"]]

    cross_coder_input = torch.stack([resid_act_A, resid_act_B], dim=0)
    cross_coder_input = cross_coder_input[:, :, 1:, :] # Drop BOS
    cross_coder_input = einops.rearrange(
        cross_coder_input,
        "n_models batch seq_len d_model -> (batch seq_len) n_models d_model",
    )

    cross_coder_output = cross_coder.decode(cross_coder.encode(cross_coder_input))
    cross_coder_output = einops.rearrange(
        cross_coder_output,
        "(batch seq_len) n_models d_model -> n_models batch seq_len d_model", batch = tokens.shape[0]
    )
    cross_coder_output_A = cross_coder_output[0]
    cross_coder_output_B = cross_coder_output[1]

    # get spliced loss
    ce_loss_spliced_A = model_A.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks = [(cross_coder.cfg["hook_point"], partial(splice_act_hook, spliced_act=cross_coder_output_A))],
    )
    ce_loss_spliced_B = model_B.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks = [(cross_coder.cfg["hook_point"], partial(splice_act_hook, spliced_act=cross_coder_output_B))],
    )

    # compute % CE recovered metric
    ce_recovered_A = 1 - ((ce_loss_spliced_A - ce_clean_A) / (ce_zero_abl_A - ce_clean_A))
    ce_recovered_B = 1 - ((ce_loss_spliced_B - ce_clean_B) / (ce_zero_abl_B - ce_clean_B))

    metrics = {
        "ce_loss_spliced_A": ce_loss_spliced_A.item(),
        "ce_loss_spliced_B": ce_loss_spliced_B.item(),
        "ce_clean_A": ce_clean_A.item(),
        "ce_clean_B": ce_clean_B.item(),
        "ce_zero_abl_A": ce_zero_abl_A.item(),
        "ce_zero_abl_B": ce_zero_abl_B.item(),
        "ce_diff_A": (ce_loss_spliced_A - ce_clean_A).item(),
        "ce_diff_B": (ce_loss_spliced_B - ce_clean_B).item(),
        "ce_recovered_A": ce_recovered_A.item(),
        "ce_recovered_B": ce_recovered_B.item(),
    }
    return metrics

tokens = all_tokens[torch.randperm(len(all_tokens))[:1]]
ce_metrics = get_ce_recovered_metrics(tokens, base_model, chat_model, folded_cross_coder)

In [ ]:
ce_metrics

For implementations of some other common evaluation metrics, like explained variance and L0, see the training codebase https://github.com/ckkissane/crosscoder-model-diff-replication

# Generating latent dashboards

Here we show how to generate latent dashboards, introduced by [Bricken et al.](https://transformer-circuits.pub/2023/monosemantic-features/vis/a1.html). We hacked a fork of [sae_vis](https://github.com/callummcdougall/sae_vis) to support crosscoders at https://github.com/ckkissane/sae_vis/tree/crosscoder-vis , which we pip install in this notebook.

In [ ]:
!pip install git+https://github.com/ckkissane/sae_vis.git@crosscoder-vis

This time we'll only fold the normalization scaling factors into W_enc, since we aren't splicing back into the model.

In [22]:
base_estimated_scaling_factor = 0.2835
chat_estimated_scaling_factor = 0.2533

import copy
folded_cross_coder = copy.deepcopy(cross_coder)

def fold_activation_scaling_factor(cross_coder, base_scaling_factor, chat_scaling_factor):
    cross_coder.W_enc.data[0, :, :] = cross_coder.W_enc.data[0, :, :] * base_scaling_factor
    cross_coder.W_enc.data[1, :, :] = cross_coder.W_enc.data[1, :, :] * chat_scaling_factor

    # cross_coder.W_dec.data[:, 0, :] = cross_coder.W_dec.data[:, 0, :] / base_scaling_factor
    # cross_coder.W_dec.data[:, 1, :] = cross_coder.W_dec.data[:, 1, :] / chat_scaling_factor

    # cross_coder.b_dec.data[0, :] = cross_coder.b_dec.data[0, :] / base_scaling_factor
    # cross_coder.b_dec.data[1, :] = cross_coder.b_dec.data[1, :] / chat_scaling_factor
    return cross_coder

folded_cross_coder = fold_activation_scaling_factor(folded_cross_coder, base_estimated_scaling_factor, chat_estimated_scaling_factor)

Here is the main boiler plate code we'll need to use the sae_vis fork. We first need to adapt our crosscoder to the forked sae_vis implementation. Then we make an SaeVisConfig and create the data with `SaeVisData.create`

In [23]:
from sae_vis.model_fns import CrossCoderConfig, CrossCoder

encoder_cfg = CrossCoderConfig(d_in=base_model.cfg.d_model, d_hidden=cross_coder.cfg["dict_size"], apply_b_dec_to_input=False)
sae_vis_cross_coder = CrossCoder(encoder_cfg)
sae_vis_cross_coder.load_state_dict(folded_cross_coder.state_dict())
sae_vis_cross_coder = sae_vis_cross_coder.to("cuda:0")
sae_vis_cross_coder = sae_vis_cross_coder.to(torch.bfloat16)

In [24]:
from sae_vis.data_config_classes import SaeVisConfig
test_feature_idx = [2325,12698,15]
sae_vis_config = SaeVisConfig(
    hook_point = folded_cross_coder.cfg["hook_point"],
    features = test_feature_idx,
    verbose = True,
    minibatch_size_tokens=4,
    minibatch_size_features=16,
)

In [ ]:
from sae_vis.data_storing_fns import SaeVisData
sae_vis_data = SaeVisData.create(
    encoder = sae_vis_cross_coder,
    encoder_B = None,
    model_A = base_model,
    model_B = chat_model,
    tokens = all_tokens[:128], # in practice, better to use more data
    cfg = sae_vis_config,
)

Finally we can view the HTML with the latent dashboards. There is a drop down in the lop left corner to view the different latents that we specified in the config.

In [ ]:
import os
import http.server
import socketserver
import threading

# Detect if running in Google Colab
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PORT = 8000  # Global port variable

def display_vis_inline(filename: str, height: int = 850):
    '''
    Displays the HTML visualization.
    - In Colab: Uses iframe.
    - In SSH: Starts a local server and prints the URL.
    '''
    global PORT

    # Use current working directory instead of hardcoded "/content"
    workspace_path = os.getcwd()  

    def serve(directory):
        os.chdir(directory)  # Change to the correct working directory

        handler = http.server.SimpleHTTPRequestHandler
        with socketserver.TCPServer(("", PORT), handler) as httpd:
            print(f"Serving files from {directory} on http://localhost:{PORT}/{filename}")
            httpd.serve_forever()

    # Start the server in a background thread
    thread = threading.Thread(target=serve, args=(workspace_path,))
    thread.daemon = True  # Allows the thread to close when the main program exits
    thread.start()

    if IN_COLAB:
        output.serve_kernel_port_as_iframe(PORT, path=f"/{filename}", height=height, cache_in_notebook=True)
    else:
        print(f"Visualization available at: http://localhost:{PORT}/{filename}")

    PORT += 1  # Increment the port for the next use

# Generate and save visualization
filename = "_feature_vis_demo.html"
sae_vis_data.save_feature_centric_vis(filename)

# Display visualization
display_vis_inline(filename)


### Matryoshka

In [ ]:
from datasets import load_dataset
def load_pile_lmsys_mixed_tokens():
    try:
        print("Loading data from disk")
        all_tokens = torch.load("/workspace/data/pile-lmsys-mix-1m-tokenized-gemma-2.pt")
    except:
        print("Data is not cached. Loading data from HF")
        data = load_dataset(
            "ckkissane/pile-lmsys-mix-1m-tokenized-gemma-2",
            split="train",
            cache_dir="/workspace/cache/"
        )
        data.save_to_disk("/workspace/data/pile-lmsys-mix-1m-tokenized-gemma-2.hf")
        data.set_format(type="torch", columns=["input_ids"])
        all_tokens = data["input_ids"]
        torch.save(all_tokens, "/workspace/data/pile-lmsys-mix-1m-tokenized-gemma-2.pt")
        print(f"Saved tokens to disk")
    return all_tokens

all_tokens = load_pile_lmsys_mixed_tokens()

In [ ]:
# 1️⃣ Load MatryoshkaCrossCoderV2

cross_coder = MatryoshkaCrossCoderV2.load_from_hf()
cross_coder.to(device)


In [11]:
# 2️⃣ Estimate or hardcode normalization factors (if not already known)
base_scaling_factor = 0.2758961493232058
chat_scaling_factor = 0.24422852496546169


In [12]:
# 2️⃣ Fold activation scaling into the crosscoder weights

def fold_scaling(cross_coder, base_factor, chat_factor):
    cross_coder.W_enc.data[0] *= base_factor
    cross_coder.W_enc.data[1] *= chat_factor
    cross_coder.W_dec.data[:, 0, :] /= base_factor
    cross_coder.W_dec.data[:, 1, :] /= chat_factor
    cross_coder.b_dec.data[0, :] /= base_factor
    cross_coder.b_dec.data[1, :] /= chat_factor
    return cross_coder

base_scaling_factor = 0.2758961493232058
chat_scaling_factor = 0.24422852496546169

cross_coder = fold_scaling(cross_coder, base_scaling_factor, chat_scaling_factor)



In [13]:
torch.cuda.empty_cache()

In [14]:
import torch, gc

# Clear CUDA memory
torch.cuda.empty_cache()
gc.collect()

# Explicitly reduce batch size
small_batch_size = 8
tokens = all_tokens[:small_batch_size].to(device)

cfg = cross_coder.cfg

# Run base model and clear memory explicitly afterwards
_, cache_base = base_model.run_with_cache(tokens, names_filter=cfg["hook_point"])
torch.cuda.empty_cache()
gc.collect()

# Run chat model and clear memory explicitly afterwards
_, cache_chat = chat_model.run_with_cache(tokens, names_filter=cfg["hook_point"])
torch.cuda.empty_cache()
gc.collect()

acts_original = torch.stack([
    cache_base[cfg["hook_point"]],
    cache_chat[cfg["hook_point"]],
], dim=1)[:, :, 1:, :]  # explicitly dropping BOS tokens

acts_original = einops.rearrange(
    acts_original,
    "batch n_models seq_len d_model -> (batch seq_len) n_models d_model"
)

acts_original[:, 0, :] *= base_scaling_factor
acts_original[:, 1, :] *= chat_scaling_factor


In [ ]:
# Explicitly convert input to match model precision
acts_original = acts_original.to(cross_coder.W_enc.dtype)

# 3️⃣ Reconstruction Quality per Nested Group
acts_encoded = cross_coder.encode(acts_original)

nested_recons = cross_coder.decode_nested(acts_encoded)

for i, rec in enumerate(nested_recons):
    # Ensure tensors match dtype
    l2_loss = (rec.float() - acts_original.float()).pow(2).mean().item()
    print(f"Nested Group {i} Reconstruction L2 Loss: {l2_loss:.6f}")


What does this mean?
Lower is better: Group 2 has the lowest loss, meaning it reconstructs the original activations the most accurately.
The improvement from Group 0 → Group 2 shows each nested group contributes progressively to a better reconstruction.

In [ ]:
# 4️⃣ Relative Decoder Norm Analysis per Nested Group

for i in range(len(cross_coder.group_sizes)):
    start, end = cross_coder.group_indices[i], cross_coder.group_indices[i+1]
    norms_group = cross_coder.W_dec[start:end].norm(dim=-1)
    relative_norms_group = norms_group[:, 1] / norms_group.sum(dim=-1)

    fig = px.histogram(relative_norms_group.cpu().numpy(),
                       title=f"Nested Group {i} Relative Decoder Norms",
                       labels={"value": "Relative Decoder Norm (IT)"})
    fig.show()



### Interpretation of Relative Decoder Norms from Above:

- **Nested Group 0**:
  - The relative norms are centered tightly around **0.5**.
  - **Meaning:** This group contributes nearly equally to both Base and Chat models, indicating that these are likely shared, fundamental features between both models.

### Nested Group 1:
- Similar tight distribution centered around 0.5.
- This also implies balanced contributions to both the base and chat models.
- Likely captures somewhat similar or overlapping semantic features as group 0.

### Nested Group 2:
- Slightly wider distribution around 0.5.
- Suggests greater variability, indicating more diverse features.
- Potentially encodes finer-grained or more model-specific features compared to groups 0 and 1.

---

### In summary:
- **Group 0 and 1** are likely encoding common, shared features of both the Base and Chat models.
- **Group 2** might begin encoding more specific nuances that slightly differentiate between the base and chat models.

Overall, all nested groups maintain a relatively balanced representation between models, but group 2 captures slightly more specialized or distinct features.

In [ ]:
batch_size, seq_len = tokens.shape

for i, rec in enumerate(nested_recons):
    # rec shape: [batch_size * (seq_len - 1), n_models, d_model]
    rec_base = rec[:, 0, :]  # select base model reconstruction
    rec_base = einops.rearrange(
        rec_base, "(batch seq) d_model -> batch seq d_model",
        batch=batch_size
    )

    # Pad at the beginning along seq dimension with zeros
    zero_pad = torch.zeros((batch_size, 1, rec_base.shape[-1]), device=rec_base.device, dtype=rec_base.dtype)
    rec_base_padded = torch.cat([zero_pad, rec_base], dim=1)

    # Now shape matches: [batch_size, seq_len, d_model]
    loss_nested = base_model.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks=[(cfg["hook_point"], lambda act, hook: rec_base_padded)]
    )
    print(f"Nested Group {i} Cross-Entropy Loss: {loss_nested.item():.6f}")


We are analyzing how each nested group in MatryoshkaCrossCoder affects the quality of reconstruction, specifically through cross-entropy loss (CE).
Each "Nested Group" refers to progressively adding more features when reconstructing activations from your cross-coder model. Specifically:

Nested Group 0: Reconstruction using only the first group of features.
Nested Group 1: Using Groups 0 and 1 combined.
Nested Group 2: Using Groups 0, 1, and 2 combined (all groups).


Observations:

Nested Group 0 (using only first group) has the highest loss.
Nested Group 1 (adding another group) decreases loss.
Nested Group 2 (all groups) further slightly decreases loss.

# Latent Dashboard Representation - Matryoshka

In [15]:
matryoshka_crosscoder = cross_coder

In [ ]:

# 🔍 SaeVis configuration (quick & efficient)
from sae_vis.data_config_classes import SaeVisConfig

quick_feature_indices = [0, 1, 2]  # visualize just a few features quickly
sae_vis_config_nested = SaeVisConfig(
    hook_point=folded_cross_coder.cfg["hook_point"],
    features=quick_feature_indices,
    verbose=True,  # clear loading bars
    minibatch_size_tokens=2,
    minibatch_size_features=4,
)

# ⚙️ Efficient SaeVis data creation (small number of tokens)
from sae_vis.data_storing_fns import SaeVisData

sae_vis_data_nested = SaeVisData.create(
    encoder=sae_vis_cross_coder_nested,
    model_A=base_model,
    model_B=chat_model,
    tokens=all_tokens[:8],  # small number for speed
    cfg=sae_vis_config_nested,
)

# 🌐 Inline visualization server function
import os, http.server, socketserver, threading

try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PORT = 8000

def display_vis_inline(filename: str, height: int = 850):
    global PORT
    workspace_path = os.getcwd()

    def serve(directory):
        os.chdir(directory)
        handler = http.server.SimpleHTTPRequestHandler
        with socketserver.TCPServer(("", PORT), handler) as httpd:
            print(f"Serving on http://localhost:{PORT}/{filename}")
            httpd.serve_forever()

    thread = threading.Thread(target=serve, args=(workspace_path,))
    thread.daemon = True
    thread.start()

    if IN_COLAB:
        output.serve_kernel_port_as_iframe(PORT, path=f"/{filename}", height=height, cache_in_notebook=True)
    else:
        print(f"Visualization available at: http://localhost:{PORT}/{filename}")

    PORT += 1

# 📊 Generate & display the visualization inline
filename = f"matryoshka_nested_group_{nested_group_idx}_feature_vis.html"
sae_vis_data_nested.save_feature_centric_vis(filename)
display_vis_inline(filename)


In [ ]:
# !ls -lh matryoshka_nested_group_0_feature_vis.html
# # if running from ssh-ed machine, you can view the html file in the browser by running:

# # from local machine
# ssh -L 8001:localhost:8001 user@remote_host

# # open the file in the browser
# open http://localhost:8001/matryoshka_nested_group_0_feature_vis.html



In [ ]:
# Quickly get top interesting features by decoder norm
decoder_norms = folded_cross_coder.W_dec.norm(dim=(1,2))
top_features = torch.topk(decoder_norms, k=10).indices.tolist()
print(f"Top 10 interesting features (by decoder norm): {top_features}")